# Import

In [3]:
import numpy as np
import pandas as pd
import optuna

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error
from scikeras.wrappers import KerasRegressor
from bayes_opt import BayesianOptimization

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import mse
from tensorflow.keras.metrics import RootMeanSquaredError, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

# 원본 데이터 전처리

In [5]:
# 3개년 데이터 로드 (인덱스 자동추가 제외)
origin_2021 = pd.read_csv('./train_subway21.csv', index_col=0)
origin_2022 = pd.read_csv('./train_subway22.csv', index_col=0)
origin_2023 = pd.read_csv('./train_subway23.csv', index_col=0)

In [6]:
# 칼럼명 변경
origin_2021 = origin_2021.rename(columns={
    'train_subway21.tm': 'time',
    'train_subway21.line': 'line',
    'train_subway21.station_number': 'station_number',
    'train_subway21.station_name': 'station_name',
    'train_subway21.direction': 'direction',
    'train_subway21.stn': 'AWS_num',
    'train_subway21.ta': 'temp',
    'train_subway21.wd': 'wind_direction',
    'train_subway21.ws': 'wind_speed',
    'train_subway21.rn_day': 'rn_day',
    'train_subway21.rn_hr1': 'rn_hr',
    'train_subway21.hm': 'humidity',
    'train_subway21.si': 'solar',
    'train_subway21.ta_chi': 'feel_temp',
    'train_subway21.congestion': 'congestion'
})

origin_2022 = origin_2022.rename(columns={
    'train_subway22.tm': 'time',
    'train_subway22.line': 'line',
    'train_subway22.station_number': 'station_number',
    'train_subway22.station_name': 'station_name',
    'train_subway22.direction': 'direction',
    'train_subway22.stn': 'AWS_num',
    'train_subway22.ta': 'temp',
    'train_subway22.wd': 'wind_direction',
    'train_subway22.ws': 'wind_speed',
    'train_subway22.rn_day': 'rn_day',
    'train_subway22.rn_hr1': 'rn_hr',
    'train_subway22.hm': 'humidity',
    'train_subway22.si': 'solar',
    'train_subway22.ta_chi': 'feel_temp',
    'train_subway22.congestion': 'congestion'
})

origin_2023 = origin_2023.rename(columns={
    'train_subway23.tm': 'time',
    'train_subway23.line': 'line',
    'train_subway23.station_number': 'station_number',
    'train_subway23.station_name': 'station_name',
    'train_subway23.direction': 'direction',
    'train_subway23.stn': 'AWS_num',
    'train_subway23.ta': 'temp',
    'train_subway23.wd': 'wind_direction',
    'train_subway23.ws': 'wind_speed',
    'train_subway23.rn_day': 'rn_day',
    'train_subway23.rn_hr1': 'rn_hr',
    'train_subway23.hm': 'humidity',
    'train_subway23.si': 'solar',
    'train_subway23.ta_chi': 'feel_temp',
    'train_subway23.congestion': 'congestion'
})


In [7]:
# origin dataframe 통합
origin_all = pd.concat([origin_2021, origin_2022, origin_2023], ignore_index=True)
origin_all

,time,line,station_number,station_name,direction,AWS_num,temp,wind_direction,wind_speed,rn_day,rn_hr,humidity,solar,feel_temp,congestion
0,2021010100,1,150,서울역,상선,419,-9.6,291.1,3.3,0.0,0.0,-99.0,-99.0,-12.6,0
1,2021010101,1,150,서울역,상선,419,-9.7,284.6,2.0,0.0,0.0,-99.0,-99.0,-9.8,0
2,2021010105,1,150,서울역,상선,419,-9.3,124.7,2.4,0.0,0.0,-99.0,-99.0,-10.3,1
3,2021010106,1,150,서울역,상선,419,-9.3,126.2,1.7,0.0,0.0,-99.0,-99.0,-10.1,2
4,2021010107,1,150,서울역,상선,419,-9.1,145.7,1.3,0.0,0.0,-99.0,-99.0,-9.7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369319,2023123119,8,2828,남위례,하선,572,0.6,0.0,0.0,7.0,0.0,83.1,-99.0,0.0,18
16369320,2023123120,8,2828,남위례,하선,572,0.0,354.7,0.0,7.0,0.0,84.7,-99.0,-0.6,17
16369321,2023123121,8,2828,남위례,하선,572,-0.6,0.0,0.0,7.0,0.0,85.1,-99.0,-1.1,21
16369322,2023123122,8,2828,남위례,하선,572,-0.8,0.0,0.0,7.0,0.0,85.6,-99.0,-1.3,18


In [8]:
# 결측값 → NaN
sentinels = [-99, -99.9, -999.0]
origin_all = origin_all.replace(sentinels, np.nan)

In [9]:
print(origin_all.isnull().sum())

time                    0
line                    0
station_number          0
station_name            0
direction               0
AWS_num                 0
temp               216468
wind_direction     230786
wind_speed         230786
rn_day             351574
rn_hr              360796
humidity           844594
solar             6064242
feel_temp             352
congestion              0
dtype: int64


In [10]:
df=origin_all.copy()

In [11]:
df = df.sort_values(['AWS_num', 'time']).reset_index(drop=True)

In [12]:
target_cols = ['temp', 'wind_direction', 'wind_speed', 'rn_day', 'rn_hr', 'humidity', 'solar', 'feel_temp']
specific_cols = ['wind_direction', 'rn_day', 'rn_hr']
linear_cols = ['temp', 'wind_speed','humidity', 'solar', 'feel_temp']

# 1단계: 관측소별 ffill,bfill로 보간방식
print('1단계시 결측치 값 변화')
for col in specific_cols:
    df[col] = df.groupby('AWS_num')[col].ffill().bfill()
print(df.isnull().sum())
print('----------------------------------')

# 2단계: 관측소별 선형보간 방식
print('2단계시 결측치 값 변화')
df[linear_cols] = df.groupby('AWS_num')[linear_cols].transform(
    lambda x: x.interpolate(method='linear')
)
print(df.isnull().sum())
print('----------------------------------')

# 3단계: 경계값 처리
print('3단계시 결측치 값 변화')
for col in linear_cols:
    df[col] = df.groupby('AWS_num')[col].ffill().bfill()
print(df.isnull().sum())
print('----------------------------------')

1단계시 결측치 값 변화
time                    0
line                    0
station_number          0
station_name            0
direction               0
AWS_num                 0
temp               216468
wind_direction          0
wind_speed         230786
rn_day                  0
rn_hr                   0
humidity           844594
solar             6064242
feel_temp             352
congestion              0
dtype: int64
----------------------------------
2단계시 결측치 값 변화
time                   0
line                   0
station_number         0
station_name           0
direction              0
AWS_num                0
temp                4634
wind_direction         0
wind_speed          4636
rn_day                 0
rn_hr                  0
humidity          633392
solar               3440
feel_temp              0
congestion             0
dtype: int64
----------------------------------
3단계시 결측치 값 변화
time              0
line              0
station_number    0
station_name      0
direction        

In [13]:
# 상/하행 레이블 정의
direction_map = {'상선': 0, '하선': 1, '내선': 2, '외선':3}

# 체감온도 구간 레이블 정의
bins = [-float('inf'), -10, 5, 20, 28, float('inf')]
labels = ["0", "1", "2", "3", "4"]

# 임시 DataFrame temp_df 정의
temp_df = df.copy()

# direction을 숫자 레이블로 변환
temp_df['direction'] = temp_df['direction'].map(direction_map)

# 체감온도 단계 파생변수 추가
temp_df['feel_stage'] = pd.cut(temp_df['feel_temp'], bins=bins, labels=labels).astype(int)

In [14]:
holidays = [
    # 2021
    '20210101','20210211','20210212','20210213','20210301',
    '20210505','20210519','20210606','20210815','20210920',
    '20210921','20210922','20211003','20211009','20211225',
    # 2022
    '20220101','20220131','20220201','20220202','20220301',
    '20220309','20220505','20220508','20220601','20220606',
    '20220815','20220909','20220910','20220911','20220912',
    '20221003','20221009','20221010','20221225',
    # 2023
    '20230101','20230121','20230122','20230123','20230124',
    '20230301','20230505','20230527','20230529','20230606',
    '20230815','20230928','20230929','20230930','20231002',
    '20231003','20231009','20231225'
]
# 공휴일 리스트를 set으로 변환하여 조회 속도 향상
holidays_set = set(holidays)

# 'time' 컬럼을 datetime 객체로 변환
temp_df['time'] = pd.to_datetime(temp_df['time'], format='%Y%m%d%H')

# 'is_holiday' 컬럼 초기화 (기본값 0: 평일)
temp_df['is_holiday'] = 0

# 주말(토/일) 확인하여 1로 설정
is_weekend = (temp_df['time'].dt.dayofweek >= 5)
temp_df.loc[is_weekend, 'is_holiday'] = 1

# 공휴일 확인하여 1로 설정
is_in_holidays = temp_df['time'].dt.strftime('%Y%m%d').isin(holidays_set)
temp_df.loc[is_in_holidays, 'is_holiday'] = 1

In [15]:
temp_df = temp_df.sort_values(['station_number', 'AWS_num','direction','time']).reset_index(drop=True)
temp_df

,time,line,station_number,station_name,direction,AWS_num,temp,wind_direction,wind_speed,rn_day,rn_hr,humidity,solar,feel_temp,congestion,feel_stage,is_holiday
0,2021-01-01 00:00:00,1,150,서울역,0,419,-9.6,291.1,3.3,0.0,0.0,47.5,0.00,-12.6,0,0,1
1,2021-01-01 01:00:00,1,150,서울역,0,419,-9.7,284.6,2.0,0.0,0.0,47.5,0.00,-9.8,0,1,1
2,2021-01-01 05:00:00,1,150,서울역,0,419,-9.3,124.7,2.4,0.0,0.0,47.5,0.00,-10.3,1,0,1
3,2021-01-01 06:00:00,1,150,서울역,0,419,-9.3,126.2,1.7,0.0,0.0,47.5,0.00,-10.1,2,0,1
4,2021-01-01 07:00:00,1,150,서울역,0,419,-9.1,145.7,1.3,0.0,0.0,47.5,0.00,-9.7,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369319,2023-12-31 19:00:00,6,9006,응암S,1,400,1.4,0.0,0.0,13.5,0.5,92.1,0.03,1.3,15,1,1
16369320,2023-12-31 20:00:00,6,9006,응암S,1,400,0.8,308.8,0.5,13.5,0.5,95.3,0.03,0.7,13,1,1
16369321,2023-12-31 21:00:00,6,9006,응암S,1,400,0.4,320.6,0.1,13.5,0.5,96.2,0.03,0.2,14,1,1
16369322,2023-12-31 22:00:00,6,9006,응암S,1,400,0.0,117.6,0.1,13.5,0.5,97.1,0.03,0.0,14,1,1


In [16]:
# 필요한 칼럼만 추출
selected_cols = [
    'time',
    'station_number',
    'direction',
    'temp',
    'wind_speed',
    'rn_hr',
    'humidity',
    'solar',
    'congestion',
    'feel_stage',
    'is_holiday'
]

# temp_df에서 필요한 컬럼만 train_data DataFrame에 저장
train_data = temp_df[selected_cols].copy()

# station_number, direction, time 순으로 정렬
train_data = train_data.sort_values(
    by=['station_number', 'direction', 'time']
).reset_index(drop=True)

# time 제거
train_data = train_data.iloc[:, 1:]

# 완성
train_data

,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage,is_holiday
0,150,0,-9.6,3.3,0.0,47.5,0.00,0,0,1
1,150,0,-9.7,2.0,0.0,47.5,0.00,0,1,1
2,150,0,-9.3,2.4,0.0,47.5,0.00,1,0,1
3,150,0,-9.3,1.7,0.0,47.5,0.00,2,0,1
4,150,0,-9.1,1.3,0.0,47.5,0.00,3,1,1
...,...,...,...,...,...,...,...,...,...,...
16369319,9006,1,1.4,0.0,0.5,92.1,0.03,15,1,1
16369320,9006,1,0.8,0.5,0.5,95.3,0.03,13,1,1
16369321,9006,1,0.4,0.1,0.5,96.2,0.03,14,1,1
16369322,9006,1,0.0,0.1,0.5,97.1,0.03,14,1,1


## 슬라이딩 윈도우

In [18]:
def extract_inputoutput_shift(df, lookback_time=2, predict_time=1):
    # 과거 시점별 피처 생성
    feat_dfs = []
    for t in range(lookback_time):
        tmp = df.shift(t)
        feat_dfs.append(tmp)
    X = pd.concat(feat_dfs, axis=1)

    # 예측 대상(타깃) 생성
    y = df[['congestion']].shift(-predict_time)

    # NaN 제거 및 인덱스 리셋
    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X = X.loc[valid_idx].reset_index(drop=True)
    y = y.loc[valid_idx].reset_index(drop=True)

    print("X, Y 데이터 분류 완료")
    return X, y

In [19]:
x, t = extract_inputoutput_shift(train_data)
x

X, Y 데이터 분류 완료


,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage,is_holiday,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage,is_holiday
0,150,0,-9.7,2.0,0.0,47.5,0.00,0,1,1,150.0,0.0,-9.6,3.3,0.0,47.5,0.00,0.0,0.0,1.0
1,150,0,-9.3,2.4,0.0,47.5,0.00,1,0,1,150.0,0.0,-9.7,2.0,0.0,47.5,0.00,0.0,1.0,1.0
2,150,0,-9.3,1.7,0.0,47.5,0.00,2,0,1,150.0,0.0,-9.3,2.4,0.0,47.5,0.00,1.0,0.0,1.0
3,150,0,-9.1,1.3,0.0,47.5,0.00,3,1,1,150.0,0.0,-9.3,1.7,0.0,47.5,0.00,2.0,0.0,1.0
4,150,0,-8.5,0.6,0.0,47.5,0.00,3,1,1,150.0,0.0,-9.1,1.3,0.0,47.5,0.00,3.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369317,9006,1,2.6,0.3,0.5,86.6,0.03,19,1,1,9006.0,1.0,4.6,0.7,0.5,76.2,0.26,18.0,1.0,1.0
16369318,9006,1,1.4,0.0,0.5,92.1,0.03,15,1,1,9006.0,1.0,2.6,0.3,0.5,86.6,0.03,19.0,1.0,1.0
16369319,9006,1,0.8,0.5,0.5,95.3,0.03,13,1,1,9006.0,1.0,1.4,0.0,0.5,92.1,0.03,15.0,1.0,1.0
16369320,9006,1,0.4,0.1,0.5,96.2,0.03,14,1,1,9006.0,1.0,0.8,0.5,0.5,95.3,0.03,13.0,1.0,1.0


In [20]:
t

,congestion
0,1.0
1,2.0
2,3.0
3,3.0
4,5.0
...,...
16369317,15.0
16369318,13.0
16369319,14.0
16369320,14.0


In [21]:
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2, shuffle=False)

print('x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

x_train shape : (13095457, 20)
t_train shape : (13095457, 1)
x_test shape : (3273865, 20)
t_test shape : (3273865, 1)


In [22]:
timesteps = 2
feature = 10

x_train = np.array(x_train)
x_train = x_train.reshape(x_train.shape[0], timesteps, feature)

x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], timesteps, feature)

t_train = np.array(t_train)
t_test = np.array(t_test)

print('reshape 후 x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('reshape 후 x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

reshape 후 x_train shape : (13095457, 2, 10)
t_train shape : (13095457, 1)
reshape 후 x_test shape : (3273865, 2, 10)
t_test shape : (3273865, 1)


In [23]:
def lstm_model(cell_size=128, learning_rate=0.001):
    timesteps = 2
    feature = 10
    model = Sequential(name='congestion_lstm_2')
    model.add(LSTM(cell_size, input_shape=(timesteps, feature), return_sequences=True,
                  kernel_regularizer=regularizers.l2(0.001)))
    model.add(LSTM(cell_size, kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss='mse', optimizer=Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
    return model

In [24]:
temp_grue = lstm_model()
temp_grue.summary()

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "congestion_lstm_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 2, 128)         │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,881 (792.50 KB)

 Trainable params: 202,881 (792.50 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# 성능 평가 함수
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

# 데이터가 너무 많아 약 20%만 추출하여 하이퍼파라미터를 탐색하도록
x_train_subset, _, t_train_subset, _ = train_test_split(
    x_train, t_train, train_size=0.2, shuffle=False , random_state=42
)

def objective(trial):
    cell_size = trial.suggest_categorical('cell_size', [64, 128])
    learning_rate = trial.suggest_float('learning_rate', 5e-4, 1e-3, log=True)
    epochs = trial.suggest_int('epochs', 10, 20)
    batch_size = trial.suggest_categorical('batch_size', [256, 512])

    # scikeras로 래핑
    Congestion_LSTM_MODEL = KerasRegressor(
        model=lstm_model,
        cell_size=cell_size,
        learning_rate=learning_rate,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping]
    )
    # cross_val_score에서 cv=2와 데이터 직접 전달
    scores = cross_val_score(Congestion_LSTM_MODEL, x_train_subset, t_train_subset, scoring=rmse_scorer, cv=2)
    return -scores.mean()

In [26]:
bayesian_study = optuna.create_study(direction='minimize')

bayesian_study.optimize(objective, n_trials=20)

[I 2025-06-17 14:05:26,198] A new study created in memory with name: no-name-ba83d571-af6c-4eac-995f-daf8cbc58786


Epoch 1/17


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2558/2558 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - loss: 604.4943 - root_mean_squared_error: 24.1884
Epoch 2/17
  34/2558 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 203.2561 - root_mean_squared_error: 14.2440

/opt/anaconda3/lib/python3.12/site-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


...

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 119.9173 - root_mean_squared_error: 10.6454
Epoch 2/10
  27/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 74.2602 - root_mean_squared_error: 8.5770  

/opt/anaconda3/lib/python3.12/site-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 63.3757 - root_mean_squared_error: 7.9277
Epoch 3/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 58.8993 - root_mean_squared_error: 7.6307
Epoch 4/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 56.0768 - root_mean_squared_error: 7.4335
Epoch 5/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 53.6721 - root_mean_squared_error: 7.2616
Epoch 6/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 52.9953 - root_mean_squared_error: 7.2077
Epoch 7/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 51.7710 - root_mean_squared_error: 7.1162
Epoch 8/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 50.6875 - root_mean_squared_error: 7.0341
Epoch 9/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 49.7386 - root_mean_squared_error: 6.9612
Epoch 10/10
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 49.2325 - root_mean_squared_error: 6.9204
5116/5116 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step


[I 2025-06-17 18:53:20,285] Trial 19 finished with value: 14.604387288790493 and parameters: {'cell_size': 128, 'learning_rate': 0.0006758121204295375, 'epochs': 10, 'batch_size': 256}. Best is trial 11 with value: 12.75138769324007.


In [27]:
print("Random Search에서의 최고의 하이퍼파라미터 조합 : ", bayesian_study.best_params)

Random Search에서의 최고의 하이퍼파라미터 조합 :  {'cell_size': 128, 'learning_rate': 0.0007746521536265282, 'epochs': 20, 'batch_size': 256}


In [28]:
best_params = bayesian_study.best_params
final_lstm_model = lstm_model(
    cell_size=best_params['cell_size'],
    learning_rate=best_params['learning_rate']
)

history = final_lstm_model.fit(
    x_train, t_train,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[early_stopping]
)

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 321s 6ms/step - loss: 106.2686 - root_mean_squared_error: 10.2081
Epoch 2/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


51155/51155 ━━━━━━━━━━━━━━━━━━━━ 301s 6ms/step - loss: 85.5827 - root_mean_squared_error: 9.1661
Epoch 3/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 320s 6ms/step - loss: 80.2262 - root_mean_squared_error: 8.8443
Epoch 4/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 318s 6ms/step - loss: 76.9557 - root_mean_squared_error: 8.6370
Epoch 5/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 316s 6ms/step - loss: 75.2132 - root_mean_squared_error: 8.5240
Epoch 6/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 320s 6ms/step - loss: 73.7317 - root_mean_squared_error: 8.4263
Epoch 7/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 317s 6ms/step - loss: 73.8392 - root_mean_squared_error: 8.4299
Epoch 8/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 304s 6ms/step - loss: 73.1968 - root_mean_squared_error: 8.3876
Epoch 9/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 313s 6ms/step - loss: 72.1967 - root_mean_squared_error: 8.3238
Epoch 10/20
51155/51155 ━━━━━━━━━━━━━━━━━━━━ 322s 6ms/step - loss: 72.1289 - root_mean_squared_error: 8.3157
Epoch 11/20
51155/51155 ━━━━━━━━━━━━━━

In [29]:
pred = final_lstm_model.predict(x_test)

for i in range(0, 20):
    print('혼잡도 예측 : ', round(pred[i][0], 2), '/정답 :', round(t_test[i][0], 2))

102309/102309 ━━━━━━━━━━━━━━━━━━━━ 59s 571us/step
혼잡도 예측 :  4.66 /정답 : 8.0
혼잡도 예측 :  8.96 /정답 : 9.0
혼잡도 예측 :  9.35 /정답 : 13.0
혼잡도 예측 :  12.87 /정답 : 18.0
혼잡도 예측 :  19.67 /정답 : 13.0
혼잡도 예측 :  9.77 /정답 : 9.0
혼잡도 예측 :  7.06 /정답 : 12.0
혼잡도 예측 :  13.43 /정답 : 5.0
혼잡도 예측 :  2.3 /정답 : 3.0
혼잡도 예측 :  3.62 /정답 : 2.0
혼잡도 예측 :  3.21 /정답 : 0.0
혼잡도 예측 :  2.44 /정답 : 6.0
혼잡도 예측 :  4.21 /정답 : 5.0
혼잡도 예측 :  3.78 /정답 : 7.0
혼잡도 예측 :  8.83 /정답 : 5.0
혼잡도 예측 :  3.3 /정답 : 3.0
혼잡도 예측 :  4.97 /정답 : 4.0
혼잡도 예측 :  1.98 /정답 : 6.0
혼잡도 예측 :  3.99 /정답 : 6.0
혼잡도 예측 :  3.96 /정답 : 6.0


In [30]:
loss, rmse = final_lstm_model.evaluate(x_test, t_test, verbose=1)
print('test loss(MSE) :', round(loss, 6))
print('test RMSE :', round(rmse, 2))

102309/102309 ━━━━━━━━━━━━━━━━━━━━ 63s 619us/step - loss: 144.2846 - root_mean_squared_error: 11.6320
test loss(MSE) : 136.410263
test RMSE : 11.53
